RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [8]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\GEO\AppData\Local\Temp\ipykernel_6096\2096081597.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [12]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/files/pdf")

Found 2 PDF files to process

Processing: 1-s2.0-S1110016823011572-main.pdf
  ✓ Loaded 12 pages

Processing: 1-s2.0-S235234092400903X-main.pdf
  ✓ Loaded 9 pages

Total documents loaded: 21


In [13]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2024-02-06T18:47:45+00:00', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'creationdate--text': '6th February 2024', 'elsevierwebpdfspecifications': '7.0', 'robots': 'noindex', 'moddate': '2024-02-06T18:55:29+00:00', 'author': 'Sherien Elkateb', 'doi': '10.1016/j.aej.2023.12.065', 'title': 'Machine learning and IoT – Based predictive maintenance approach for industrial applications', 'keywords': 'Failure,IoT,Knitting machines,Machine learning,Predictive maintenance', 'subject': 'Alexandria Engineering Journal, 88 (2024) 298-309. doi:10.1016/j.aej.2023.12.065', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'source': '..\\data\\files\\pdf\\1-s2.0-S1110016823011572-main.pdf', 'total_pages': 12, 'page': 0, 'page_label': '298', 'source_file': '1-s2.0-S1110016823011572-main.pdf', 'file_type': 'pdf'}, page_content='A

In [14]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [15]:
chunks=split_documents(all_pdf_documents)
chunks 

Split 21 documents into 99 chunks

Example chunk:
Content: Alexandria Engineering Journal 88 (2024) 298–309
Available online 20 January 2024
1110-0168/© 2024 The Author(s). Published by Elsevier BV on behalf of Faculty of Engineering, Alexandria University Th...
Metadata: {'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2024-02-06T18:47:45+00:00', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'creationdate--text': '6th February 2024', 'elsevierwebpdfspecifications': '7.0', 'robots': 'noindex', 'moddate': '2024-02-06T18:55:29+00:00', 'author': 'Sherien Elkateb', 'doi': '10.1016/j.aej.2023.12.065', 'title': 'Machine learning and IoT – Based predictive maintenance approach for industrial applications', 'keywords': 'Failure,IoT,Knitting machines,Machine learning,Predictive maintenance', 'subject': 'Alexandria Engineering Journal, 88 (2024) 298-309. doi:10.1016/j.aej.2023.12.065', 'crossmarkdomains[2]': 'sciencedir

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2024-02-06T18:47:45+00:00', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'creationdate--text': '6th February 2024', 'elsevierwebpdfspecifications': '7.0', 'robots': 'noindex', 'moddate': '2024-02-06T18:55:29+00:00', 'author': 'Sherien Elkateb', 'doi': '10.1016/j.aej.2023.12.065', 'title': 'Machine learning and IoT – Based predictive maintenance approach for industrial applications', 'keywords': 'Failure,IoT,Knitting machines,Machine learning,Predictive maintenance', 'subject': 'Alexandria Engineering Journal, 88 (2024) 298-309. doi:10.1016/j.aej.2023.12.065', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'source': '..\\data\\files\\pdf\\1-s2.0-S1110016823011572-main.pdf', 'total_pages': 12, 'page': 0, 'page_label': '298', 'source_file': '1-s2.0-S1110016823011572-main.pdf', 'file_type': 'pdf'}, page_content='A

embedding And vectorStoreDB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
from typing import List
import numpy as np

In [1]:
import os

os.environ["HF_HOME"] = os.path.abspath(".hf_cache")

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\GEO\Downloads\RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GEO\Downloads\RAG\notebook\.hf_cache\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7483.86it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\GEO\AppData\Local\Temp\ipykernel_6096\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


VectorStore

In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [16]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'Elsevier', 'creationdate': '2024-02-06T18:47:45+00:00', 'crossmarkdomains[1]': 'elsevier.com', 'crossmarkmajorversiondate': '2010-04-23', 'creationdate--text': '6th February 2024', 'elsevierwebpdfspecifications': '7.0', 'robots': 'noindex', 'moddate': '2024-02-06T18:55:29+00:00', 'author': 'Sherien Elkateb', 'doi': '10.1016/j.aej.2023.12.065', 'title': 'Machine learning and IoT – Based predictive maintenance approach for industrial applications', 'keywords': 'Failure,IoT,Knitting machines,Machine learning,Predictive maintenance', 'subject': 'Alexandria Engineering Journal, 88 (2024) 298-309. doi:10.1016/j.aej.2023.12.065', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkdomainexclusive': 'true', 'source': '..\\data\\files\\pdf\\1-s2.0-S1110016823011572-main.pdf', 'total_pages': 12, 'page': 0, 'page_label': '298', 'source_file': '1-s2.0-S1110016823011572-main.pdf', 'file_type': 'pdf'}, page_content='A

In [17]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store into the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 99 texts...


Batches: 100%|██████████| 4/4 [00:04<00:00,  1.00s/it]


Generated embeddings with shape: (99, 384)
Adding 99 documents to vector store...
Successfully added 99 documents to vector store
Total documents in collection: 99


Retriever Pipeline From VectorStore

In [18]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [19]:
rag_retriever

In [20]:
rag_retriever.retrieve("What is predictive maintenance?")

Retrieving documents for query: 'What is predictive maintenance?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.73it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d384168c_57',
  'content': 'Lycra stops. By enabling timely maintenance actions, the system has the \ncapability to enhance overall efficiency and productivity within the \ntextile industry. Implementing the proposed predictive maintenance \nsystem holds significant advantages for the textile industry such as \nmitigating machine failures, reducing unplanned downtime, minimizing \nproduction losses, and reducing maintenance costs. Moreover, it em-\npowers businesses to adopt a proactive maintenance approach, resulting \nin improved operational efficiency and resource optimization. Accord-\ningly, this study has significant potential impact in the textile industry. It \nenhances machine lifetime, improves product quality, and improves \nmanufacturer income. Future work includes expanding the data inputs \nby more stop types to cover all incipient faults. Moreover, the presented \ntechnique will be applied on different types of machines. \nFunding \nSTDF grant number 43467. 

In [21]:
rag_retriever.retrieve("Machine Learning techniques for predictive maintenance in industrial equipment")

Retrieving documents for query: 'Machine Learning techniques for predictive maintenance in industrial equipment'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_21f5c33b_1',
  'content': 'd Electrical Engineering Department, Faculty of Engineering, Alexandria University, Egypt   \nARTICLE INFO  \nKeywords: \nFailure \nIoT \nKnitting machines \nMachine learning \nPredictive maintenance \nABSTRACT  \nUnplanned outage in industry due to machine failures can lead to significant production losses and increased \nmaintenance costs. Predictive maintenance methods use the data collected from IoT-enabled devices installed in \nworking machines to detect incipient faults and prevent major failures. In this study, a predictive maintenance \nsystem based on machine learning algorithms, specifically AdaBoost, is presented to classify different types of \nmachines stops in real-time with application in knitting machines. The data collected from the machines include \nmachine speeds and steps, which were pre-processed and fed into the machine learning model to classify six \ntypes of machines stops: gate stop, feeder stop, needle stop, completed

RAG Pipeline- VectorDB To LLM Output Generation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

In [24]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [25]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"
    

In [35]:
# Initialize Groq LLM
try:
    groq_llm = GroqLLM(
        model_name="openai/gpt-oss-20b",
        api_key=os.getenv("GROQ_API_KEY")
    )

    print("Groq LLM initialized successfully!")

except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-20b
Groq LLM initialized successfully!


In [29]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Predictive Maintenance in industrial equipment")

Retrieving documents for query: 'Predictive Maintenance in industrial equipment'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  8.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_d384168c_57',
  'content': 'Lycra stops. By enabling timely maintenance actions, the system has the \ncapability to enhance overall efficiency and productivity within the \ntextile industry. Implementing the proposed predictive maintenance \nsystem holds significant advantages for the textile industry such as \nmitigating machine failures, reducing unplanned downtime, minimizing \nproduction losses, and reducing maintenance costs. Moreover, it em-\npowers businesses to adopt a proactive maintenance approach, resulting \nin improved operational efficiency and resource optimization. Accord-\ningly, this study has significant potential impact in the textile industry. It \nenhances machine lifetime, improves product quality, and improves \nmanufacturer income. Future work includes expanding the data inputs \nby more stop types to cover all incipient faults. Moreover, the presented \ntechnique will be applied on different types of machines. \nFunding \nSTDF grant number 43467. 

Enhanced RAG Pipeline Features

In [36]:
def rag_advanced(
    query,
    retriever,
    llm,
    top_k=5,
    min_score=0.2,
    return_context=False
):
    """
    Enhanced RAG pipeline with extra features:
    - Retrieves relevant documents
    - Filters documents using similarity score
    - Generates an answer using the Groq LLM
    - Returns sources
    - Calculates confidence score
    - Optionally returns the full retrieved context
    """

    
    # 1. Retrieve relevant documents
   
    results = retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=min_score
    )

    # 2. Handle case where no relevant documents are found
    if not results:
        output = {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0
        }

        if return_context:
            output['context'] = ''

        return output

   
    # 3. Prepare context from retrieved documents
    context = "\n\n".join(
        [doc['content'] for doc in results]
    )

    # 4. Prepare source information
    sources = []

    for doc in results:

        metadata = doc.get('metadata', {})

        source = metadata.get(
            'source_file',
            metadata.get('source', 'unknown')
        )

        page = metadata.get(
            'page',
            'unknown'
        )

        score = doc.get(
            'similarity_score',
            0.0
        )

        content = doc.get(
            'content',
            ''
        )

        sources.append({
            'source': source,
            'page': page,
            'score': score,
            'preview': (
                content[:300] + '...'
                if len(content) > 300
                else content
            )
        })

    # 5. Calculate confidence score
    scores = [
        doc.get('similarity_score', 0.0)
        for doc in results
    ]

    if scores:
        confidence = max(scores)
    else:
        confidence = 0.0

   
    # 6. Generate answer using your GroqLLM class
    answer = llm.generate_response(
        query=query,
        context=context
    )


    # 7. Build final output
    output = {
        'answer': answer,
        'sources': sources,
        'confidence': confidence
    }

    # 8. Optionally include retrieved context
    if return_context:
        output['context'] = context

    return output


result = rag_advanced(
    query="Predictive Maintenance",
    retriever=rag_retriever,
    llm=groq_llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)



# Display Results

print("\n" + "=" * 60)
print("ANSWER")
print("=" * 60)

print(result['answer'])


print("\n" + "=" * 60)
print("SOURCES")
print("=" * 60)

for source in result['sources']:
    print(f"Source: {source['source']}")
    print(f"Page: {source['page']}")
    print(f"Similarity Score: {source['score']}")
    print(f"Preview: {source['preview']}")
    print("-" * 60)


print("\n" + "=" * 60)
print("CONFIDENCE")
print("=" * 60)

print(result['confidence'])


print("\n" + "=" * 60)
print("CONTEXT PREVIEW")
print("=" * 60)

if result.get('context'):
    print(result['context'][:1000])
else:
    print("No context available.")

Retrieving documents for query: 'Predictive Maintenance'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.69it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)



ANSWER
**Predictive Maintenance in the Textile Industry**

Predictive maintenance (PdM) uses data‑driven techniques—such as machine‑learning models, sensor analytics, and fault‑diagnosis algorithms—to forecast when a machine is likely to fail or require servicing. In the context of textile manufacturing, especially on circular knitting machines, PdM aims to:

| What it does | Why it matters |
|--------------|----------------|
| **Detect incipient faults** (e.g., abnormal vibration, temperature, or motor current) | Prevents sudden stoppages that halt production lines. |
| **Estimate Remaining Useful Life (RUL)** of critical components | Enables scheduled, rather than reactive, interventions. |
| **Diagnose fault types** (e.g., motor slip, belt wear, sensor drift) | Allows targeted repairs, reducing unnecessary part replacements. |
| **Generate maintenance alerts** in real time | Gives operators actionable information before a breakdown occurs. |

### Key Benefits Highlighted in the Con

In [42]:
# Advanced RAG Pipeline: Citations, History, Summarization

from typing import List, Dict, Any


class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        """
        Initialize the Advanced RAG Pipeline.

        Args:
            retriever: Your RAG retriever.
            llm: Your GroqLLM object.
        """

        self.retriever = retriever
        self.llm = llm
        self.history = []


    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # 1. Retrieve relevant documents

        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        # 2. Handle no results

        if not results:

            answer = "No relevant context found."
            sources = []
            context = ""

        else:

            # 3. Prepare context

            context = "\n\n".join(
                [doc["content"] for doc in results]
            )

            # 4. Prepare sources

            sources = []

            for doc in results:

                metadata = doc.get("metadata", {})

                source = metadata.get(
                    "source_file",
                    metadata.get("source", "unknown")
                )

                page = metadata.get(
                    "page",
                    "unknown"
                )

                score = doc.get(
                    "similarity_score",
                    0.0
                )

                content = doc.get(
                    "content",
                    ""
                )

                sources.append({
                    "source": source,
                    "page": page,
                    "score": score,
                    "preview": (
                        content[:120] + "..."
                        if len(content) > 120
                        else content
                    )
                })

            # 5. Generate answer

            if stream:
                print("\nGenerating answer...\n")

            answer = self.llm.generate_response(
                query=question,
                context=context
            )

            if stream:
                print(answer)
                print()

        # 6. Add citations

        if sources:

            citations = "\n".join(
                [
                    f"[{i + 1}] "
                    f"{src['source']} "
                    f"(page {src['page']})"
                    for i, src in enumerate(sources)
                ]
            )

            answer_with_citations = (
                answer
                + "\n\nCitations:\n"
                + citations
            )

        else:
            answer_with_citations = answer

        # 7. Optional summarization

        summary = None

        if summarize and answer:

            summary_context = (
                f"Summarize the following answer "
                f"in 2 concise sentences:\n\n{answer}"
            )

            summary = self.llm.generate_response(
                query="Summarize the answer in 2 concise sentences.",
                context=summary_context
            )

        # 8. Store query history

        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        # 9. Return results

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }


# Create Advanced RAG Pipeline

adv_rag = AdvancedRAGPipeline(
    retriever=rag_retriever,
    llm=groq_llm
)


# Example Query

result = adv_rag.query(
    question="What is Predictive Maintenance?",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)


# Display Final Results

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(result["answer"])


print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

print(result["summary"])


print("\n" + "=" * 60)
print("SOURCES")
print("=" * 60)

for source in result["sources"]:

    print(f"Source: {source['source']}")
    print(f"Page: {source['page']}")
    print(f"Score: {source['score']}")
    print(f"Preview: {source['preview']}")
    print("-" * 60)


print("\n" + "=" * 60)
print("QUERY HISTORY")
print("=" * 60)

print(result["history"][-1])

Retrieving documents for query: 'What is Predictive Maintenance?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

Generating answer...



**Predictive Maintenance** is a proactive approach that uses data‑driven techniques—such as sensor monitoring, machine‑learning algorithms, and statistical analysis—to forecast when a machine is likely to fail or require service. By predicting stoppages, remaining useful life (RUL), or specific fault types before they occur, the system enables timely maintenance actions. This reduces unplanned downtime, cuts maintenance costs, extends machine life, and improves overall operational efficiency and product quality in industries such as textile manufacturing.


FINAL ANSWER
**Predictive Maintenance** is a proactive approach that uses data‑driven techniques—such as sensor monitoring, machine‑learning algorithms, and statistical analysis—to forecast when a machine is likely to fail or require service. By predicting stoppages, remaining useful life (RUL), or specific fault types before they occur, the system enables timely maintenance actions. This reduces unplanned downtime, cuts maintenance